In [ ]:
!pip install mediapipe==0.10.13

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 18.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.


In [ ]:
import os
import cv2
import random
import numpy as np
import tensorflow as tf
import mediapipe as mp
import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam


In [ ]:
path = kagglehub.dataset_download("hasyimabdillah/workoutfitness-video")
DATA_DIR = path


100%|██████████| 4.32G/4.32G [03:26<00:00, 22.5MB/s]

Extracting files...


In [ ]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

def extract_keypoints(results):
    """Extract pose landmarks as flat array"""
    if results.pose_landmarks:
        keypoints = []
        for landmark in results.pose_landmarks.landmark:
            keypoints.extend([landmark.x, landmark.y,
                            landmark.z, landmark.visibility])
        return np.array(keypoints)
    return np.zeros(33 * 4)

In [ ]:
def process_video(video_path, label):
    """Extract pose sequences from a video"""
    cap = cv2.VideoCapture(str(video_path))
    sequences = []

    with mp_pose.Pose(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Convert to RGB
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            # Detect pose
            results = pose.process(image)

            # Extract keypoints
            keypoints = extract_keypoints(results)
            sequences.append(keypoints)

    cap.release()

    # Create dataframe
    df = pd.DataFrame(sequences)
    df['label'] = label
    df['frame'] = range(len(sequences))

    return df

In [ ]:

data_dir = Path(DATA_DIR)
all_data = []

exercises = [
        'barbell biceps curl', 'bench press', 'chest fly machine',
        'deadlift', 'decline bench press', 'hammer curl',
        'hip thrust', 'incline bench press', 'lat pulldown',
        'lateral raises', 'leg extension', 'leg raises',
        'plank', 'pull up', 'push up', 'romanian deadlift',
        'russian twist', 'shoulder press', 'squat',
        't bar row', 'tricep dips', 'tricep pushdown'
    ]

for exercise in tqdm(exercises):
  exercise_dir = data_dir / exercise
  if not exercise_dir.exists():
    print(f"Skipping: {exercise} (not found)")
    continue

  for video_file in exercise_dir.glob('*.mp4'):
    print(f"Processing: {video_file.name}")
    df = process_video(video_file, exercise)
    all_data.append(df)


  0%|          | 0/22 [00:00<?, ?it/s]

Processing: barbell biceps curl_29.mp4


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing: barbell biceps curl_27.mp4


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


...

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing: tricep dips_17.mp4


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 22/22 [47:19<00:00, 129.05s/it]

Skipping: tricep pushdown (not found)


OSError: Cannot save file into a non-existent directory: 'data/processed'

In [ ]:
final_df = pd.concat(all_data, ignore_index=True)
final_df.to_csv('raw_poses.csv', index=False)
print(f"Extracted {len(final_df)} frames from all videos")


Extracted 87586 frames from all videos


In [ ]:
def calculate_angle(a, b, c):
    """
    Calculate angle between three points
    a, b, c: [x, y, z] coordinates
    Returns: angle in degrees
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - \
              np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:

def calculate_distance(a, b):
    """Euclidean distance between two points"""
    a = np.array(a)
    b = np.array(b)
    return np.linalg.norm(a - b)

In [ ]:



def extract_features(landmarks):
    """
    Extract meaningful features from pose landmarks
    landmarks: array of shape (33, 4) - 33 keypoints
    """
    # Reshape landmarks
    lm = landmarks.reshape(33, 4)

    features = []

    # Key joint angles
    # Right arm
    right_elbow = calculate_angle(
        lm[12][:3], lm[14][:3], lm[16][:3]  # shoulder-elbow-wrist
    )
    features.append(right_elbow)

    # Left arm
    left_elbow = calculate_angle(
        lm[11][:3], lm[13][:3], lm[15][:3]
    )
    features.append(left_elbow)

    # Right leg
    right_knee = calculate_angle(
        lm[24][:3], lm[26][:3], lm[28][:3]  # hip-knee-ankle
    )
    features.append(right_knee)

    # Left leg
    left_knee = calculate_angle(
        lm[23][:3], lm[25][:3], lm[27][:3]
    )
    features.append(left_knee)

    # Hip angle (torso bend)
    right_hip = calculate_angle(
        lm[12][:3], lm[24][:3], lm[26][:3]  # shoulder-hip-knee
    )
    features.append(right_hip)

    left_hip = calculate_angle(
        lm[11][:3], lm[23][:3], lm[25][:3]
    )
    features.append(left_hip)

    # Shoulder angles
    right_shoulder = calculate_angle(
        lm[24][:3], lm[12][:3], lm[14][:3]  # hip-shoulder-elbow
    )
    features.append(right_shoulder)

    left_shoulder = calculate_angle(
        lm[23][:3], lm[11][:3], lm[13][:3]
    )
    features.append(left_shoulder)

    # Distances
    # Hand to opposite knee
    right_hand_left_knee = calculate_distance(
        lm[16][:3], lm[25][:3]
    )
    features.append(right_hand_left_knee)

    left_hand_right_knee = calculate_distance(
        lm[15][:3], lm[26][:3]
    )
    features.append(left_hand_right_knee)

    # Hands distance
    hands_distance = calculate_distance(
        lm[15][:3], lm[16][:3]
    )
    features.append(hands_distance)

    # Body center height (hip average y-coordinate)
    body_height = (lm[23][1] + lm[24][1]) / 2
    features.append(body_height)

    # Torso width
    torso_width = calculate_distance(
        lm[11][:3], lm[12][:3]
    )
    features.append(torso_width)

    # Visibility scores (important for detecting occlusion)
    avg_visibility = np.mean(lm[:, 3])
    features.append(avg_visibility)

    return np.array(features)




In [ ]:
def create_temporal_features(df, window_size=30):
    """
    Create features from sequence of frames
    window_size: number of frames to look back (1 second at 30fps)
    """
    temporal_features = []
    labels = []

    for label in tqdm(df['label'].unique()):
        label_data = df[df['label'] == label]

        # Get raw pose data (columns 0-131 are the 33*4 keypoints)
        pose_cols = [col for col in df.columns
                     if col not in ['label', 'frame']]
        poses = label_data[pose_cols].values

        # Sliding window
        for i in range(len(poses) - window_size):
            window = poses[i:i+window_size]

            # Extract features for each frame in window
            frame_features = []
            for frame_poses in window:
                feats = extract_features(frame_poses)
                frame_features.append(feats)

            frame_features = np.array(frame_features)

            # Aggregate temporal features
            features = []

            # Statistical features
            features.extend(np.mean(frame_features, axis=0))
            features.extend(np.std(frame_features, axis=0))
            features.extend(np.max(frame_features, axis=0))
            features.extend(np.min(frame_features, axis=0))

            # Velocity (change over time)
            velocities = np.diff(frame_features, axis=0)
            features.extend(np.mean(velocities, axis=0))
            features.extend(np.std(velocities, axis=0))

            temporal_features.append(features)
            labels.append(label)

    return np.array(temporal_features), np.array(labels)


In [ ]:
df=pd.read_csv('raw_poses.csv')
print("Creating temporal features...")
X, y = create_temporal_features(df, window_size=30)



Creating temporal features...


100%|██████████| 18/18 [04:57<00:00, 16.53s/it]


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/X.npy'

In [ ]:
np.save('X.npy', X)
np.save('y.npy', y)

print(f"Feature shape: {X.shape}")
print(f"Labels shape: {y.shape}")

Feature shape: (87046, 84)
Labels shape: (87046,)


In [ ]:
np.save('X.npy', X)
np.save('y.npy', y)

print(f"Feature shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import pickle

def train_xgboost():
    # Load processed features
    X = np.load('X.npy')
    y = np.load('y.npy')

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42,
        stratify=y_encoded
    )

    print(f"Training samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")

    # Train XGBoost
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    print("Training model...")
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = (y_pred == y_test).mean()

    print(f"\nTest Accuracy: {accuracy:.2%}")
    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred,
        target_names=le.classes_
    ))

    # Save model and encoder
    with open('pose_classifier.pkl', 'wb') as f:
        pickle.dump(model, f)

    with open('label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)

    print("\nModel saved!")

    return model, le



In [ ]:
train_xgboost()

Training samples: 69636
Test samples: 17410
Training model...

Test Accuracy: 99.53%

Classification Report:
                     precision    recall  f1-score   support

barbell biceps curl       1.00      1.00      1.00      1403
        bench press       1.00      1.00      1.00      1211
  chest fly machine       1.00      1.00      1.00      1008
           deadlift       1.00      1.00      1.00      1020
decline bench press       1.00      0.93      0.96       309
        hammer curl       1.00      1.00      1.00       511
         hip thrust       1.00      1.00      1.00       816
incline bench press       1.00      1.00      1.00      1415
       lat pulldown       1.00      0.98      0.99      1249
      leg extension       1.00      0.98      0.99       711
         leg raises       1.00      1.00      1.00       631
              plank       1.00      1.00      1.00      1751
  romanian deadlift       0.93      1.00      0.96       986
      russian twist       1.00      

(XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.8, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=0.1, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=200,
               n_jobs=-1, num_parallel_tree=None, ...),
 LabelEncoder())

In [ ]:
import cv2
from collections import deque

with open('pose_classifier.pkl', 'rb') as f:
    model = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

# Initialize MediaPipe
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# Temporal smoothing
WINDOW_SIZE = 30  # 1 second at 30fps
pose_buffer = deque(maxlen=WINDOW_SIZE)
prediction_buffer = deque(maxlen=10)

def extract_keypoints(results):
    """Extract pose landmarks"""
    if results.pose_landmarks:
        keypoints = []
        for landmark in results.pose_landmarks.landmark:
            keypoints.extend([landmark.x, landmark.y,
                            landmark.z, landmark.visibility])
        return np.array(keypoints)
    return np.zeros(33 * 4)

def get_smoothed_prediction(buffer):
    """Mode of recent predictions"""
    if len(buffer) < 5:
        return None
    return max(set(buffer), key=buffer.count)



In [ ]:
cap = cv2.VideoCapture(0)

with mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as pose:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

            keypoints = extract_keypoints(results)
            pose_buffer.append(keypoints)

            if len(pose_buffer) == WINDOW_SIZE:
                window = np.array(pose_buffer)
                frame_features = []

                for frame_poses in window:
                    feats = extract_features(frame_poses)
                    frame_features.append(feats)

                frame_features = np.array(frame_features)

                features = []
                features.extend(np.mean(frame_features, axis=0))
                features.extend(np.std(frame_features, axis=0))
                features.extend(np.max(frame_features, axis=0))
                features.extend(np.min(frame_features, axis=0))

                velocities = np.diff(frame_features, axis=0)
                features.extend(np.mean(velocities, axis=0))
                features.extend(np.std(velocities, axis=0))

                features = np.array(features).reshape(1, -1)

                pred = model.predict(features)[0]
                proba = model.predict_proba(features)[0]
                confidence = proba[pred]

                if confidence > 0.7:
                    prediction_buffer.append(pred)

                smoothed_pred = get_smoothed_prediction(
                    prediction_buffer
                )

                if smoothed_pred is not None:
                    exercise = label_encoder.inverse_transform(
                        [smoothed_pred]
                    )[0]

                    cv2.putText(
                        image,
                        f"Exercise: {exercise}",
                        (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (0, 255, 0),
                        2
                    )
                    cv2.putText(
                        image,
                        f"Confidence: {confidence:.2%}",
                        (10, 70),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (0, 255, 0),
                        2
                    )

        cv2.imshow("Exercise Detection", image)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()
